## Deep Dyna-Q Tutorial with CartPole

Adapted from [Deep Dyna-Q: Integrating Planning for Task-Completion Dialogue Policy Learning](https://arxiv.org/abs/1801.06176) (Peng et al., 2018, Microsoft Research). The agent learns a world model from real transitions and uses it to generate simulated experience, supplementing real interactions — both contribute to the Q-network's gradients.

The original paper was coupled to a dialogue NLP task. This implementation generalizes the core algorithm to work with any Gymnasium environment.

![Alt text](best_run.png)

In [8]:
import os
from collections.abc import Callable
from typing import Any

import gymnasium as gym
import numpy as np
import structlog
import torch
import torch.nn as nn
from cart_pole_logger_dyna import CartPoleLoggerDyna
from cart_pole_wm_interpreter import CartPoleInterpreter

from samsara_rl.control.dyna.deep_dyna_q import (
    DeepDynaQ,
)
from samsara_rl.functions.neural_networks.fully_connected import FullyConnected
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [9]:
cart_pole_wm_interpreter = CartPoleInterpreter()

In [ ]:
log = structlog.get_logger()

MAX_EPISODES = 2000
BASE_DIR = "logs/deep_dyna_q"

ALPHA = 0.0005
GAMMA = 0.99
MODEL_PLAN_STEPS = [0, 2, 5]
WARMUPS = [300, 500, 800]

## Q-Network Architecture

Input: 4 values (position, velocity, angle, angular velocity). Output: 2 Q-values (push left, push right).

Two hidden layers (32 → 16) capture the nonlinear relationship between angle and angular velocity. A single hidden layer can work but requires more careful tuning — the angle/angular velocity dynamics involve sine/cosine curves that benefit from the extra representational capacity.

In [4]:
class CartPoleQNetwork(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim_1: int,
        hidden_dim_2: int,
        output_dim: int,
        preprocess: Callable[..., torch.Tensor] | None = None,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1, dtype=torch.float64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2, dtype=torch.float64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, output_dim, dtype=torch.float64)
        self.preprocess = preprocess

    def forward(self, x: Any) -> torch.Tensor:
        out: torch.Tensor = self.preprocess(x) if self.preprocess else x
        if isinstance(out, np.ndarray):
            out = torch.from_numpy(out).to(torch.float64)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu2(out)
        return self.fc3(out)  # type: ignore[no-any-return]

In [ ]:
def run_experiment(env, model_plan_steps: int, warmup: int) -> None:
    """Train a Deep Dyna-Q agent with fixed hyperparameters, sweeping plan steps and warmup.

    Args:
        env: Gymnasium CartPole environment.
        model_plan_steps: Number of Q-network training steps on simulated data per real transition.
        warmup: Number of episodes before planning begins.
    """
    model_name = "deep_dyna_q"
    run_name = f"{model_name}_plan_steps={model_plan_steps}_warmup={warmup}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)
    cart_pole_wm = FullyConnected(6, 32, 6)
    q = CartPoleQNetwork(4, 32, 16, 2)

    agent = DeepDynaQ(
        mdp=env,
        world_model=cart_pole_wm,
        world_model_interpreter=cart_pole_wm_interpreter,
        model_plan_steps=model_plan_steps,
        count_planning_updates=True,
        model_train_steps=10,
        warmup=warmup,
        gamma=GAMMA,
        q=q,
        alpha=ALPHA,
        state_loss=torch.nn.MSELoss(),
        target_update_freq=500,
        epsilon_episodes=800,
        epsilon_min=0.01,
        lr_step_size=100,
        lr_gamma=0.3,
        log_dir=run_dir,
        experiment_name=run_name,
        batch_size=128,
        buffer_size=10000,
    )
    agent.register(CartPoleLoggerDyna())
    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [ ]:
def main() -> None:
    """Sweep over plan steps and warmup periods with fixed hyperparameters."""
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for plan_steps in MODEL_PLAN_STEPS:
        for warmup in WARMUPS:
            run_experiment(env, plan_steps, warmup)

In [7]:
main()

2026-10-26 12:37:17 [info     ] starting_run                   run='deep_dyna_q_alpha=0.0001_gamma=0.99epsilon_decay=0.9995_step_2'


/home/ashish/reinforcement-learning-library/samsara-rl/src/samsara_rl/control/dyna/deep_dyna_q.py:92: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.q_scheduler.step()
/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. 

2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.6571885353063887
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=10.475293532525816
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.679227842149367
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=6.975675796419592
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.613060977633317
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.6905782630859196
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.92500943767587
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=3.895897641191599
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=0.38850110672234445
2026-10-26 12:37:23 [info     ] world_model_loss               episode=20 loss=13.676296712654596
2026-10-26 12:37:23 [info

2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=5.574666879144028
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=6.253117850050037
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=0.3475299469102137
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=5.2166146717440975
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=8.351661679332533
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=2.996525480193367
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=5.133065802600399
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=3.262226681341465
2026-10-26 12:37:42 [info     ] world_model_loss               episode=60 loss=0.6760340139769565
2026-10-26 12:37:51 [info     ] world_model_loss               episode=80 loss=11.513770794818312
2026-10-26 12:37:51 [info 

2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=5.057411526716018
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=0.7685419279323105
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=2.3044566055302393
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=2.3062769661880114
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=4.987035912065938
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=5.857160565000109
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=0.9065939426134287
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=1.029265633023588
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=2.4794363385224654
2026-10-26 12:38:02 [info     ] world_model_loss               episode=100 loss=7.745350846327574
2026-10-26 12:3

...

2026-10-27 01:16:02 [info     ] world_model_loss               episode=2700 loss=0.057164044592911065
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.07305620530504181
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=3.2530934047235682
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.019901955743622288
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.01854857114271366
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.005711899909079322
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.007899397915892002
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.018184688580211127
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=3.0476072963663827
2026-10-27 01:16:03 [info     ] world_model_loss               episode=2700 loss=0.00136

2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.015489635189117393
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.006652695127089775
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.0778866354429479
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.003952537592131589
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.0023968090432128202
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=3.0465121437829388
2026-10-27 01:16:08 [info     ] world_model_loss               episode=2700 loss=0.0035911906373846296
2026-10-27 01:16:09 [info     ] world_model_loss               episode=2700 loss=0.07407091554684044
2026-10-27 01:16:09 [info     ] world_model_loss               episode=2700 loss=0.0025012634778519647
2026-10-27 01:16:09 [info     ] world_model_loss               episode=2700 loss=0.0

2026-10-27 01:16:12 [info     ] world_model_loss               episode=2700 loss=0.003707085788968675
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.03254193758818272
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.002711557867054467
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.013736576730980984
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.007649559506025809
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=3.3227678046699487
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.003634990211817607
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.0025189929837806297
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.06277180190867623
2026-10-27 01:16:13 [info     ] world_model_loss               episode=2700 loss=0.15

2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=0.01277167464210331
2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=2.7201820365197644
2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=0.08274858048118601
2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=0.0011986463379514407
2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=0.00629192354442347
2026-10-27 01:16:17 [info     ] world_model_loss               episode=2700 loss=0.01676893533148416
2026-10-27 01:16:18 [info     ] world_model_loss               episode=2700 loss=0.007413169717099408
2026-10-27 01:16:18 [info     ] world_model_loss               episode=2700 loss=0.040833056007812264
2026-10-27 01:16:18 [info     ] world_model_loss               episode=2700 loss=0.0051939977026348675
2026-10-27 01:16:19 [info     ] world_model_loss               episode=2700 loss=0.044

KeyboardInterrupt: 

## Analysis 

### Policy and Q-Surface after 50 Episodes
![Alt text](initial_policy.png)

After 50 episodes the agent learns the correct convex shape — states closer to upright are worth more, with the origin as the peak. However the relative difference in Q-values between actions is heavily skewed. It can't distinguish when to push left vs right.

### Policy and Q-Surface after 800 Episodes
![Alt text](policy.png)

After 800 episodes the action split converges close to optimal — push left when leaning right, push right when leaning left. Learning rate decay was crucial; without it the max(Q, a') step causes Q-values to spiral due to [maximization bias](https://en.wikipedia.org/wiki/Reinforcement_learning#Maximization_bias).

### No Alpha Decay — Maximization Bias
![Alt text](no_alpha_decay.png)

Without aggressive LR decay, Q-estimates spiral and one action dominates everywhere. The tabletop shape of the Q-surface shows the network treats all non-edge states as equally good — a well-trained network would have gradual curvature. The fix is either decaying the learning rate or tuning gamma to compress the future reward range enough for the agent to distinguish good from bad states.